In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Tensor-Netzwerk-Fehlerminderung (TEM): Eine Qiskit Function von Algorithmiq
*Siehe die [API-Referenz](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*

> **Note:** Qiskit Functions sind ein experimentelles Feature, das ausschließlich Nutzerinnen und Nutzern des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) zur Verfügung steht. Sie befinden sich im Preview-Status und können sich noch ändern.


<Accordion>
<AccordionItem title="Paketversionen">

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
## Überblick
Die Tensor-Netzwerk-Fehlerminderungsmethode (TEM) von Algorithmiq ist ein hybrides
quanten-klassisches Verfahren, das Rauschminderung vollständig in der klassischen
Nachverarbeitungsphase durchführt. Mit TEM kannst du Erwartungswerte von Observablen berechnen
und dabei die unvermeidlichen rauschbedingten Fehler auf Quantenhardware mit
verbesserter Genauigkeit und Kosteneffizienz mindern – eine attraktive Option
für Quantenforscher und Industriepraktiker gleichermaßen.

Die Methode besteht darin, ein Tensor-Netzwerk zu konstruieren, das die Inverse des
globalen Rauschkanals darstellt, der den Zustand des Quantenprozessors beeinflusst,
und diese Abbildung dann auf informationsvollständige Messergebnisse anzuwenden,
die aus dem verrauschten Zustand gewonnen wurden, um unverzerrte Schätzer für die
Observablen zu erhalten.

Als Vorteil nutzt TEM informationsvollständige Messungen, um Zugang zu einer
großen Menge gemilderter Erwartungswerte von Observablen zu erhalten, und hat
optimalen Sampling-Overhead auf der Quantenhardware, wie in Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), und Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542) beschrieben. Der
Mess-Overhead bezieht sich auf die Anzahl zusätzlicher Messungen, die für eine
effiziente Fehlerminderung erforderlich sind – ein entscheidender Faktor für die
Durchführbarkeit von Quantenberechnungen. Daher hat TEM das Potenzial, einen
Quantenvorteil in komplexen Szenarien zu ermöglichen, etwa bei Anwendungen in den
Bereichen Quantenchaos, Vielteilchenphysik, Hubbard-Dynamik und Simulationen
kleiner Molekülchemie.

Die wichtigsten Merkmale und Vorteile von TEM lassen sich wie folgt zusammenfassen:

1.  **Optimaler Mess-Overhead**: TEM ist optimal hinsichtlich der
[theoretischen Schranken](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
d. h. keine andere Methode kann einen geringeren Mess-Overhead erzielen. Anders
gesagt benötigt TEM die minimale Anzahl zusätzlicher Messungen für die
Fehlerminderung. Das bedeutet wiederum, dass TEM die Quantenlaufzeit minimiert.
2.  **Kosteneffizienz**: Da TEM die Rauschminderung vollständig in der
Nachverarbeitungsphase übernimmt, müssen dem Quantencomputer keine zusätzlichen
Circuits hinzugefügt werden. Das macht die Berechnung nicht nur günstiger,
sondern verringert auch das Risiko, durch die Unvollkommenheiten von
Quantengeräten zusätzliche Fehler einzuführen.
3.  **Schätzung mehrerer Observablen**: Dank informationsvollständiger
Messungen schätzt TEM effizient mehrere Observablen mit denselben
Messdaten vom Quantencomputer.
4.  **Messfehlerminderung**: Die TEM Qiskit Function enthält außerdem eine
[proprietäre Methode zur Messfehlerminderung](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154),
die Auslesfehler nach einem kurzen Kalibrierungslauf deutlich reduzieren kann.
5.  **Genauigkeit**: TEM verbessert die Genauigkeit und Zuverlässigkeit
digitaler Quantensimulationen erheblich und macht Quantenalgorithmen präziser
und verlässlicher.
## Beschreibung
Die TEM-Funktion ermöglicht es dir, fehlergeminderte Erwartungswerte für
mehrere Observablen eines Quantum Circuits mit minimalem Sampling-Overhead zu berechnen.
Der Circuit wird mit einer informationsvollständigen positiven operatorwertigen
Maßnahme (IC-POVM) gemessen, und die gesammelten Messergebnisse werden auf einem
klassischen Computer verarbeitet. Diese Messung wird verwendet, um die
Tensor-Netzwerk-Methoden durchzuführen und eine rausch-invertierende Abbildung
aufzubauen. Die Funktion wendet eine Abbildung an, die den gesamten verrauschten
Circuit vollständig invertiert, indem Tensor-Netzwerke zur Darstellung der
verrauschten Schichten verwendet werden.

![TEM-Schemata](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Fehlergeminderte Schätzung einer Observablen O durch Nachverarbeitung der Messergebnisse des verrauschten Quantenprozessors. U und N bezeichnen eine ideale Quantenoperation und die zugehörige Rauschkarte, die im Allgemeinen nicht-lokal sein kann (und auf graue Boxen ausgedehnt wird). D steht für einen Tensor von Operatoren, die dual zu den Effekten der IC-Messung sind. Das Rauschminderungsmodul M ist ein Tensor-Netzwerk, das effizient von der Mitte nach außen kontrahiert wird. Die erste Iteration der Kontraktion wird durch die gepunktete lila Linie dargestellt, die zweite durch die gestrichelte Linie und die dritte durch die durchgezogene Linie.")

Sobald die Circuits an die Funktion übermittelt werden, werden sie transpiliert und
optimiert, um die Anzahl der Schichten mit Zwei-Qubit-Gates zu minimieren (den
rauschintensiveren Gates auf Quantengeräten). Das die Schichten beeinflussende
Rauschen wird über
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
mithilfe eines spärlichen Pauli-Lindblad-Rauschmodells erlernt, wie in E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023) beschrieben.
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Das Rauschmodell ist eine genaue Beschreibung des Rauschens auf dem Gerät und kann
subtile Merkmale erfassen, einschließlich Qubit-Übersprechen. Allerdings kann das
Rauschen auf den Geräten schwanken und driften, sodass das erlernte Rauschmodell
zum Zeitpunkt der Schätzung möglicherweise nicht mehr akkurat ist. Das kann zu
ungenauen Ergebnissen führen.
## Erste Schritte
Authentifiziere dich mit deinem [IBM Quantum Platform API-Schlüssel](http://quantum.cloud.ibm.com/) und wähle die TEM-Funktion wie folgt aus. (Dieser Codeausschnitt setzt voraus, dass du dein Konto bereits [in deiner lokalen Umgebung gespeichert hast](/guides/functions#install-qiskit-functions-catalog-client).)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")
# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load your function
tem = catalog.load(tem_function_name)

Verwende die Qiskit Serverless APIs, um den Status deines Qiskit Function-Workloads zu überprüfen:

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Use the Qiskit Serverless APIs to check your Qiskit Function workload's status:

In [3]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())

30db31ed-d252-4ca8-a5f0-5eb9b5075a4c
QUEUED


Du kannst die Ergebnisse wie folgt abrufen:

In [4]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

0.09417719217134088


<Admonition type="info">
  The expected value for the noiseless circuit for the given operator should be around `0.18409094298943401`.
</Admonition>

## Get support

Reach out to [qiskit_ibm@algorithmiq.fi](mailto:qiskit_ibm@algorithmiq.fi)

Be sure to include the following information:
- Qiskit Function Job ID (`qiskit-ibm-catalog`), `job.job_id`
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Algorithmiq Tensor-network error mitigation](https://quantum.ibm.com/functions?id=4b1b9d76-c18b-4788-b70b-15125111fbe6).
- Visit the [API reference](/docs/api/functions/algorithmiq-tem) for this Qiskit Function.

</Admonition>